In [1]:
import pandas as pd
import numpy as np

# Load the training and testing datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the training and testing datasets
print("Training Data Sample:")
print(train_df.head())
print("\nTesting Data Sample:")
print(test_df.head())

# Display the basic information of the training and testing datasets
print("\nTraining Data Info:")
print(train_df.info())
print("\nTesting Data Info:")
print(test_df.info())

# Display the summary statistics of the training and testing datasets
print("\nTraining Data Summary Statistics:")
print(train_df.describe())
print("\nTesting Data Summary Statistics:")
print(test_df.describe())

# Distinguish column types for tailored analysis and visualization
print("\nTraining Data Column Types:")
print(train_df.select_dtypes(include=['object']).columns)
print("\nTraining Data Numerical Columns:")
print(train_df.select_dtypes(include=[np.number]).columns)

print("\nTesting Data Column Types:")
print(test_df.select_dtypes(include=['object']).columns)
print("\nTesting Data Numerical Columns:")
print(test_df.select_dtypes(include=[np.number]).columns)

# Check for missing values in the training and testing datasets
print("\nMissing Values in Training Data:")
print(train_df.isnull().sum())
print("\nMissing Values in Testing Data:")
print(test_df.isnull().sum())

# Check for duplicate rows in the training and testing datasets
print("\nDuplicate Rows in Training Data:")
print(train_df.duplicated().sum())
print("\nDuplicate Rows in Testing Data:")
print(test_df.duplicated().sum())


Training Data Sample:
        Category  DayOfWeek PdDistrict           X          Y
0  LARCENY/THEFT  Wednesday   SOUTHERN -122.388380  37.783310
1   NON-CRIMINAL   Saturday   SOUTHERN -122.403405  37.775421
2  LARCENY/THEFT  Wednesday   NORTHERN -122.419581  37.789214
3  VEHICLE THEFT     Friday    BAYVIEW -122.389744  37.757909
4        ASSAULT     Friday    TARAVAL -122.478377  37.742877

Testing Data Sample:
         Category  DayOfWeek  PdDistrict           X          Y
0   VEHICLE THEFT     Sunday    SOUTHERN -122.409893  37.780113
1        BURGLARY     Monday    NORTHERN -122.424442  37.788227
2  OTHER OFFENSES  Wednesday     BAYVIEW -122.395635  37.753565
3        BURGLARY    Tuesday        PARK -122.444802  37.754171
4    NON-CRIMINAL   Saturday  TENDERLOIN -122.412971  37.785788

Training Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70244 entries, 0 to 70243
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      -------------

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Assuming 'train_df' is the DataFrame from the 'Finished Tasks'
column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-15 07:54:32.895 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Category', 'DayOfWeek', 'PdDistrict'], 'Numeric': ['X', 'Y'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Copy the DataFrames to avoid modifying the original data
train_df_processed = train_df.copy()
test_df_processed = test_df.copy()

# Handle missing values
fill_missing = FillMissingValue(features=['Category', 'DayOfWeek', 'PdDistrict', 'X', 'Y'], strategy='most_frequent')
train_df_processed = fill_missing.fit_transform(train_df_processed)
test_df_processed = fill_missing.transform(test_df_processed)

# Encode categorical variables
label_encode = LabelEncode(features=['Category', 'DayOfWeek', 'PdDistrict'])
train_df_processed = label_encode.fit_transform(train_df_processed)
test_df_processed = label_encode.transform(test_df_processed)

# Normalize numerical features
standard_scale = StandardScale(features=['X', 'Y'])
train_df_processed = standard_scale.fit_transform(train_df_processed)
test_df_processed = standard_scale.transform(test_df_processed)

# Display the processed data
print("Processed Training Data Sample:")
print(train_df_processed.head())
print("\nProcessed Testing Data Sample:")
print(test_df_processed.head())


Processed Training Data Sample:
   Category  DayOfWeek  PdDistrict         X         Y
0        16          6           7  1.307195  0.077780
1        20          2           7  0.736721  0.038043
2        16          6           4  0.122545  0.107514
3        34          0           0  1.255416 -0.050156
4         1          0           8 -2.109862 -0.125866

Processed Testing Data Sample:
   Category  DayOfWeek  PdDistrict         X         Y
0        34          3           7  0.490380  0.061678
1         4          1           4 -0.062032  0.102543
2        21          6           0  1.031712 -0.072036
3         4          5           5 -0.835081 -0.068983
4        20          2           9  0.373523  0.090260


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info_train = get_column_info(train_df_processed)
column_info_test = get_column_info(test_df_processed)

print("Training Data Column Info:")
print(column_info_train)

print("\nTesting Data Column Info:")
print(column_info_test)


Training Data Column Info:
{'Category': [], 'Numeric': ['Category', 'DayOfWeek', 'PdDistrict', 'X', 'Y'], 'Datetime': [], 'Others': []}

Testing Data Column Info:
{'Category': [], 'Numeric': ['Category', 'DayOfWeek', 'PdDistrict', 'X', 'Y'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

# Assuming the processed data is already loaded from previous tasks
train_df_processed = train_df.copy()
test_df_processed = test_df.copy()

# Separate features and target variable
X_train = train_df_processed.drop('Category', axis=1)
y_train = train_df_processed['Category']

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Initialize and train the XGBoost classifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=10, verbose=True)

# Predict on the validation set
y_val_pred = xgb.predict_proba(X_val)

# Calculate log loss on the validation set
val_log_loss = log_loss(y_val, y_val_pred)
print(f"Validation Log Loss: {val_log_loss}")

# Predict on the test set
test_df_processed = test_df_processed.drop('Category', axis=1)
y_test_pred = xgb.predict_proba(test_df_processed)

# Save the predictions to a CSV file
predictions_df = pd.DataFrame(y_test_pred, columns=[f'Category_{i}' for i in range(y_test_pred.shape[1])])
predictions_df.to_csv('crime_category_predictions.csv', index=False)
print("Predictions saved to 'crime_category_predictions.csv'")


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'